# Marine OCR Layout Classifier Training

Phase 2 notebook: fine-tune EfficientNet-B0 on `training_data/` and export `model.onnx`.

Expected class folders:
- `spare_parts_table`
- `drawing_material_list`
- `drawing_only`
- `index_page`
- `repair_kit`

In [ ]:
!pip -q install onnx onnxruntime scikit-learn

from pathlib import Path
import copy
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

DATA_DIR = Path('/content/training_data')
MODEL_OUT = Path('/content/model.onnx')
BATCH_SIZE = 32
EPOCHS = 12
LR = 3e-4
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

Upload/copy your rendered `training_data/` folder to `/content/training_data` before running the next cell. If using Google Drive, mount Drive and point `DATA_DIR` to that path.

In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomRotation(degrees=2),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

base_dataset = datasets.ImageFolder(DATA_DIR)
class_names = base_dataset.classes
print('Classes:', class_names)
print('Images:', len(base_dataset))
print('Class counts:', {class_names[i]: base_dataset.targets.count(i) for i in range(len(class_names))})

assert set(class_names) == {'spare_parts_table', 'drawing_material_list', 'drawing_only', 'index_page', 'repair_kit'}, class_names

In [ ]:
targets = np.array(base_dataset.targets)
train_indices = []
val_indices = []
rng = np.random.default_rng(SEED)

for cls_idx in range(len(class_names)):
    cls_indices = np.where(targets == cls_idx)[0]
    rng.shuffle(cls_indices)
    split = max(1, int(0.8 * len(cls_indices)))
    if len(cls_indices) > 1 and split == len(cls_indices):
        split -= 1
    train_indices.extend(cls_indices[:split].tolist())
    val_indices.extend(cls_indices[split:].tolist())

train_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(DATA_DIR, transform=val_transform)
train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('Train images:', len(train_subset))
print('Val images:', len(val_subset))

In [ ]:
model = efficientnet_b0(weights=weights)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_state = None
best_val_acc = 0.0

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, True)
    val_loss, val_acc = run_epoch(val_loader, False)
    scheduler.step()
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
    print(f'Epoch {epoch:02d}: train_loss={train_loss:.4f} train_acc={train_acc:.3f} val_loss={val_loss:.4f} val_acc={val_acc:.3f}')

model.load_state_dict(best_state)
print('Best val accuracy:', best_val_acc)

In [ ]:
model.eval()
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in val_loader:
        logits = model(images.to(device))
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

print(classification_report(all_labels, all_preds, target_names=class_names, digits=4, zero_division=0))
cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(class_names))))
print('Confusion matrix rows=true cols=pred:')
print(cm)

per_class_accuracy = {}
for idx, name in enumerate(class_names):
    denom = cm[idx].sum()
    per_class_accuracy[name] = float(cm[idx, idx] / denom) if denom else 0.0
print('Per-class validation accuracy:', per_class_accuracy)

weak_classes = [name for name, acc in per_class_accuracy.items() if acc < 0.85]
if weak_classes:
    print('FLAG: these classes are below 85% validation accuracy and need more labeled data before pipeline wiring:', weak_classes)
else:
    print('All classes are >= 85% validation accuracy on this split.')

In [ ]:
model.eval().cpu()
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model,
    dummy,
    MODEL_OUT,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)

metadata = {'class_names': class_names, 'input_size': 224, 'preprocessing': 'resize_256_center_crop_224_imagenet'}
Path('/content/model_metadata.json').write_text(json.dumps(metadata, indent=2))
print('Exported:', MODEL_OUT)
print('Exported metadata: /content/model_metadata.json')